# Baseline v5

### 필요 라이브러리 설치

In [49]:
%pip install pandas numpy scikit-learn lightgbm

Note: you may need to restart the kernel to use updated packages.


### LightGBM 모델 학습 및 자체 평가

In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. 데이터 불러오기
print("데이터 로딩 중... (용량이 커서 몇 초 정도 걸릴 수 있습니다)")
train = pd.read_csv('./data/train.csv')

# 2. 결측치 처리 (이전에 계획한 단순 대체 방법)
print("결측치 처리 중...")
# 숫자형 결측치는 평균으로, 범주형 결측치는 'Unknown'으로 채웁니다.
numeric_cols = train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if train[col].isnull().any():
        train[col] = train[col].fillna(0)

object_cols = train.select_dtypes(include=['object']).columns
for col in object_cols:
    if train[col].isnull().any():
        train[col] = train[col].fillna('Unknown')

# 3. Train / Validation 데이터 나누기 (매우 중요!)
print("학습용/검증용 데이터 분리 중...")
# 대회 평가에 제출할 'row_id'는 예측에 방해되므로 제외하고, 정답지인 'control_success'도 분리합니다.
X = train.drop(columns=['control_success', 'row_id'])
y = train['control_success']

# 모델 학습용 80%, 로컬 검증용 20%로 나눕니다.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42 # 항상 동일하게 나뉘도록 시드 고정
)

print(f"✅ 완료! 학습용 데이터 크기: {X_train.shape}, 검증용 데이터 크기: {X_valid.shape}")

데이터 로딩 중... (용량이 커서 몇 초 정도 걸릴 수 있습니다)
결측치 처리 중...
학습용/검증용 데이터 분리 중...


/var/folders/hm/v0t_41rn1k745bstq7qk0tn40000gn/T/ipykernel_1575/3981364984.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = train.select_dtypes(include=['object']).columns


✅ 완료! 학습용 데이터 크기: (1180073, 47), 검증용 데이터 크기: (295019, 47)


In [51]:
import lightgbm as lgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
import numpy as np
import pandas as pd
import joblib
import os

os.makedirs('./model', exist_ok=True)

print("1. Trackman 물리 데이터 및 구종 구사 비율 장착 중...")
# 트랙맨 데이터 불러오기 (구종 그룹 열 추가)
trackman = pd.read_csv('./data/trackman_history.csv', usecols=['pitcher_trackman_id', 'rel_speed', 'spin_rate', 'pitch_type_group'])

# [기존] 구속, 회전수 평균
pitcher_physics = trackman.groupby('pitcher_trackman_id')[['rel_speed', 'spin_rate']].mean()
speed_dict = pitcher_physics['rel_speed'].to_dict()
spin_dict = pitcher_physics['spin_rate'].to_dict()
mean_speed = trackman['rel_speed'].mean()
mean_spin = trackman['spin_rate'].mean()

# 🌟 [NEW] 제1구종 의존도 비율 계산
pitch_counts = trackman.groupby('pitcher_trackman_id').size()
# 투수별 가장 많이 던진 구종의 카운트
top_pitch_counts = trackman.groupby('pitcher_trackman_id')['pitch_type_group'].apply(lambda x: x.value_counts().max())
primary_pitch_ratio = top_pitch_counts / pitch_counts

primary_pitch_dict = primary_pitch_ratio.to_dict()
mean_primary_ratio = primary_pitch_ratio.mean()

# 파일로 저장 (서버 제출용)
joblib.dump(speed_dict, './model/speed_dict.pkl')
joblib.dump(spin_dict, './model/spin_dict.pkl')
joblib.dump(mean_speed, './model/mean_speed.pkl')
joblib.dump(mean_spin, './model/mean_spin.pkl')

joblib.dump(primary_pitch_dict, './model/primary_pitch_dict.pkl')
joblib.dump(float(mean_primary_ratio), './model/mean_primary_ratio.pkl')

# Train/Valid 데이터에 맵핑
X_train['avg_speed'] = X_train['pitcher_id'].map(speed_dict).fillna(mean_speed)
X_train['avg_spin'] = X_train['pitcher_id'].map(spin_dict).fillna(mean_spin)
X_train['primary_pitch_ratio'] = X_train['pitcher_id'].map(primary_pitch_dict).fillna(mean_primary_ratio)

X_valid['avg_speed'] = X_valid['pitcher_id'].map(speed_dict).fillna(mean_speed)
X_valid['avg_spin'] = X_valid['pitcher_id'].map(spin_dict).fillna(mean_spin)
X_valid['primary_pitch_ratio'] = X_valid['pitcher_id'].map(primary_pitch_dict).fillna(mean_primary_ratio)

print("1.5 야구 상황 기반 파생 변수 생성 중...")
def add_baseball_features(df):
    df = df.copy() 
    df['pressure_index'] = 0
    df['pressure_index'] += np.where(df['inning'] >= 7, 1, 0)
    df['pressure_index'] += np.where(df['balls_before'] == 3, 2, 0)
    df['pressure_index'] += np.where((df['outs_before'] == 2) & (df['balls_before'] >= 2), 1, 0)
    
    df['count_advantage'] = df['strikes_before'] - df['balls_before']
    df['platoon_advantage'] = np.where(df['pitcher_hand'] == df['batter_hand'], 1, 0)
    return df

X_train = add_baseball_features(X_train)
X_valid = add_baseball_features(X_valid)

print("2. 범주형 데이터 변환 중...")
object_cols = X_train.select_dtypes(include=['object']).columns
for col in object_cols:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')

print("3. LightGBM + 확률 보정(Calibration) 모델 학습 시작...")
base_model = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
model.fit(X_train, y_train)

joblib.dump(model, './model/model.pkl')

print("\n4. 모의고사 채점 중...")
y_pred_proba = model.predict_proba(X_valid)[:, 1]
brier = brier_score_loss(y_valid, y_pred_proba)
overall_mean = y_train.mean()
bss = max(0, 100000 * (1 - brier / (overall_mean * (1 - overall_mean))))

print(f"🎯 V4 최종 평가 환산 점수: {bss:.2f}점")

1. Trackman 물리 데이터 및 구종 구사 비율 장착 중...
1.5 야구 상황 기반 파생 변수 생성 중...
2. 범주형 데이터 변환 중...
3. LightGBM + 확률 보정(Calibration) 모델 학습 시작...


/var/folders/hm/v0t_41rn1k745bstq7qk0tn40000gn/T/ipykernel_1575/2560516872.py:65: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = X_train.select_dtypes(include=['object']).columns


[LightGBM] [Info] Number of positive: 412201, number of negative: 374514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015832 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6403
[LightGBM] [Info] Number of data points in the train set: 786715, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.523952 -> initscore=0.095882
[LightGBM] [Info] Start training from score 0.095882
[LightGBM] [Info] Number of positive: 412201, number of negative: 374514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016033 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6400
[LightGBM] [Info] Number of data points in the train set: 786715, number of used features: 50
[LightGBM] [In

### 학습 모델 저장

In [52]:
import joblib
import os

# model 폴더가 없다면 생성합니다.
os.makedirs('./model', exist_ok=True)

# 아까 학습시킨 LightGBM 모델을 'model.pkl'이라는 이름으로 저장합니다.
joblib.dump(model, './model/model.pkl')
print("모델이 ./model/model.pkl 경로에 성공적으로 저장되었습니다")

모델이 ./model/model.pkl 경로에 성공적으로 저장되었습니다


In [53]:
# 수정 후
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("lightgbm\nscikit-learn\n")
print("requirements.txt 파일이 성공적으로 생성되었습니다")

requirements.txt 파일이 성공적으로 생성되었습니다


### 제출용 script.py 생성

In [54]:
script_content = """import pandas as pd
import numpy as np
import joblib
import os

def main():
    test = pd.read_csv('./data/test.csv')
    row_ids = test['row_id']
    X_test = test.drop(columns=['row_id'])
    
    # Trackman 물리 데이터(속도, 회전수) 장착
    speed_dict = joblib.load('./model/speed_dict.pkl')
    spin_dict = joblib.load('./model/spin_dict.pkl')
    mean_speed = joblib.load('./model/mean_speed.pkl')
    mean_spin = joblib.load('./model/mean_spin.pkl')
    
    primary_pitch_dict = joblib.load('./model/primary_pitch_dict.pkl')
    mean_primary_ratio = joblib.load('./model/mean_primary_ratio.pkl')
    
    X_test['avg_speed'] = X_test['pitcher_id'].map(speed_dict).fillna(mean_speed)
    X_test['avg_spin'] = X_test['pitcher_id'].map(spin_dict).fillna(mean_spin)
    X_test['primary_pitch_ratio'] = X_test['pitcher_id'].map(primary_pitch_dict).fillna(mean_primary_ratio)
    
    # 🌟 [NEW] 야구 도메인 파생 변수 추가 블록
    X_test['pressure_index'] = 0
    X_test['pressure_index'] += np.where(X_test['inning'] >= 7, 1, 0)
    X_test['pressure_index'] += np.where(X_test['balls_before'] == 3, 2, 0)
    X_test['pressure_index'] += np.where((X_test['outs_before'] == 2) & (X_test['balls_before'] >= 2), 1, 0)
    X_test['count_advantage'] = X_test['strikes_before'] - X_test['balls_before']
    X_test['platoon_advantage'] = np.where(X_test['pitcher_hand'] == X_test['batter_hand'], 1, 0)
    
    # 결측치 처리 (0으로 다시 통일하여 베이스라인 안정성 확보)
    numeric_cols = X_test.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if X_test[col].isnull().any():
            X_test[col] = X_test[col].fillna(0)
            
    object_cols = X_test.select_dtypes(include=['object']).columns
    for col in object_cols:
        if X_test[col].isnull().any():
            X_test[col] = X_test[col].fillna('Unknown')
        X_test[col] = X_test[col].astype('category')
        
    model = joblib.load('./model/model.pkl')
    pred_proba = model.predict_proba(X_test)[:, 1]
    
    os.makedirs('./output', exist_ok=True)
    pd.DataFrame({
        'row_id': row_ids,
        'control_success': pred_proba
    }).to_csv('./output/submission.csv', index=False)

if __name__ == '__main__':
    main()
"""

with open("script.py", "w", encoding="utf-8") as f:
    f.write(script_content)
print("✅ V3.3 제출용 script.py 생성 완료!")

✅ V3.3 제출용 script.py 생성 완료!
